# 04 - Semantic Analysis

## Intro

In questo capitolo andremo a sfruttare il machine learning per mappare prima il testo di ogni articolo, poi le singole parole in modo da poter analizzare e la somiglianza tematica e narrativa degli articoli e l'evoluzione del linguaggio (semantic shift).

In particolare la prima parte di dividerà in questo modo:
- Aggregazione del testo: unione di headline, abstract e lead_paragraph
- NLP processing: pulizia del testo 
- Distintività lessicale per periodo (log-odds)
- Word2Vec
- Topic modeling con BERTopic


## 0. Setup

Controlliamo se abbiamo tutte le librerie e le dipendenze che ci servono e definiamo i path su cui andremo poi a lavorare.

In [ ]:
import importlib.util
import logging
import subprocess
import sys
from pathlib import Path

# import_name -> pip_name (spaCy/scikit-learn: nome import diverso dal nome pip)
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "nltk": "nltk",
    "spacy": "spacy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "gensim": "gensim",
    "bertopic": "bertopic",
    "sentence_transformers": "sentence-transformers",
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print(f"Installazione pacchetti mancanti: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Tutti i pacchetti richiesti sono gia' installati.")

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import spacy

%matplotlib inline

# Dati NLTK necessari per pulizia testo (tokenizzazione, stopword, lemmatizzazione)
for resource, path in [
    ("punkt", "tokenizers/punkt"),
    ("punkt_tab", "tokenizers/punkt_tab"),
    ("stopwords", "corpora/stopwords"),
    ("wordnet", "corpora/wordnet"),
]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(resource, quiet=True)

# Modello spaCy inglese (corpus NYT) — se assente, download on-demand
SPACY_MODEL = "en_core_web_sm"
if not importlib.util.find_spec(SPACY_MODEL):
    print(f"Download modello spaCy '{SPACY_MODEL}'...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", SPACY_MODEL])
nlp = spacy.load(SPACY_MODEL)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"

IN_ICE = PROCESSED_DIR / "articles_ice.parquet"

sns.set_theme(style="whitegrid", palette="muted")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("semantic_analysis")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")


## 1. Aggregazione del testo 

In [ ]:
log.info(f"Caricamento dati da {IN_ICE.name}...")
df = pd.read_parquet(IN_ICE)

# Sostituisco i valori NaN con stringhe vuote per evitare errori di tokenizzazione
for col in ["headline", "abstract", "lead_paragraph"]:
    df[col] = df[col].fillna('')

df['semantic_text'] = (
    df['headline'] + ' ' +
    df['abstract'] + ' ' + 
    df['lead_paragraph'] 
    )

# Rimuovo eventuali spazi multipli e spazi iniziali/finali causati dalla concatenazione dei campi
df['semantic_text'] = df['semantic_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Esempio di visualizzazione dei primi articoli
print(f"Dataset caricato: {len(df)} articoli.")
print("Esempio di articoli (campo 'semantic_text'):")
print("-" * 50)
print(df[['semantic_text']].head(n=3))
# Periodi usati in tutto il notebook (stessi confini dei notebook 02 e 03)
df['pub_date'] = pd.to_datetime(df['pub_date'], utc=True)

def get_period(date):
    if date.year <= 2016:
        return "2003-2016"
    elif date.year <= 2020:
        return "2017-2020"
    elif date.year <= 2024:
        return "2021-2024"
    else:
        return "2025-Presente"

df['periodo'] = df['pub_date'].apply(get_period)

palette = {
    "2003-2016": "#1f77b4",
    "2017-2020": "#d62728",
    "2021-2024": "#2ca02c",
    "2025-Presente": "#9467bd",
}


## 2. Pulizia 

In [ ]:
clean_docs = []

for doc in nlp.pipe(df['semantic_text'], disable=["ner", "parser"]):
    tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    clean_docs.append(" ".join(tokens))

df['clean_text'] = clean_docs

tokens_prima = df['semantic_text'].apply(lambda x: len(x.split())).sum()
tokens_dopo = df['clean_text'].apply(lambda x: len(x.split())).sum()
riduzione = (1 - (tokens_dopo / tokens_prima)) * 100

print(f"\n--- COMPRESSIONE DATASET ---")
print(f"Volume originale:  {tokens_prima:,} parole")
print(f"Volume ripulito:   {tokens_dopo:,} lemmi")

print("\nPRIMA (Testo Aggregato):")
print(df['semantic_text'].iloc[0])

print("\nDOPO (Testo Pulito - Lemmatizzato):")
print(df['clean_text'].iloc[0])


## 3. Distintività lessicale per periodo (log-odds con prior di Dirichlet)

Una media TF-IDF per periodo premierebbe le parole frequenti **in tutti** i periodi (*immigration, say, new, year*): dice poco su cosa distingue un periodo dagli altri.

Qui usiamo il **log-odds ratio con prior di Dirichlet informativo** (Monroe, Colaresi & Quinn 2008, *Fightin' Words*), lo standard per confrontare il lessico di due gruppi di testi ed estrarre le parole più significative per periodo.

- per ogni parola confronta la sua frequenza nel periodo con quella nel resto del corpus;
- il prior (le frequenze dell'intero corpus, peso `alpha0`) smorza le parole rare, che altrimenti avrebbero rapporti estremi per puro caso;
- il risultato è uno **z-score**: |z| > 1.96 ≈ differenza significativa al 5%.

Lavoriamo su `semantic_text` (non lemmatizzato), unigrammi e bigrammi: i bigrammi catturano formule come *illegal immigrants* vs *undocumented immigrants*, che la lemmatizzazione e i singoli token spezzerebbero.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

LOGODDS_STOPWORDS = list(ENGLISH_STOP_WORDS | {"said", "says", "mr", "ms", "new", "year", "years", "old"})
LOGODDS_ALPHA0 = 1000  # peso totale del prior: più alto = parole rare più penalizzate

logodds_vectorizer = CountVectorizer(
    stop_words=LOGODDS_STOPWORDS,
    ngram_range=(1, 2),
    min_df=5,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",  # esclude numeri e token di una lettera
)
counts_matrix = logodds_vectorizer.fit_transform(df['semantic_text'])
logodds_vocab = logodds_vectorizer.get_feature_names_out()


def log_odds_dirichlet(counts, mask_a, mask_b, alpha0=LOGODDS_ALPHA0) -> pd.DataFrame:
    """z-score del log-odds ratio (gruppo A vs gruppo B) con prior dalle frequenze dell'intero corpus."""
    y_a = np.asarray(counts[mask_a].sum(axis=0)).ravel()
    y_b = np.asarray(counts[mask_b].sum(axis=0)).ravel()
    y_all = np.asarray(counts.sum(axis=0)).ravel()
    prior = alpha0 * y_all / y_all.sum()
    n_a, n_b = y_a.sum(), y_b.sum()
    delta = (np.log((y_a + prior) / (n_a + alpha0 - y_a - prior))
             - np.log((y_b + prior) / (n_b + alpha0 - y_b - prior)))
    variance = 1 / (y_a + prior) + 1 / (y_b + prior)
    return pd.DataFrame({"term": logodds_vocab, "z": delta / np.sqrt(variance), "count_a": y_a, "count_b": y_b})


PERIODI = sorted(df['periodo'].unique())
N_DISTINCTIVE = 15

distinctive = {}
for periodo in PERIODI:
    mask = (df['periodo'] == periodo).values
    distinctive[periodo] = log_odds_dirichlet(counts_matrix, mask, ~mask).sort_values("z", ascending=False)


In [ ]:
fig, axes = plt.subplots(1, len(PERIODI), figsize=(22, 7), sharex=False)
for ax, periodo in zip(axes, PERIODI):
    top = distinctive[periodo].head(N_DISTINCTIVE).iloc[::-1]
    ax.barh(top['term'], top['z'], color=palette[periodo])
    ax.axvline(1.96, color='grey', linestyle=':', linewidth=1)
    ax.set_title(f"{periodo}\n({(df['periodo'] == periodo).sum()} articoli)", fontsize=13, fontweight='bold')
    ax.set_xlabel("z-score (periodo vs resto)")
plt.suptitle("Parole e bigrammi più distintivi di ogni periodo (log-odds, prior di Dirichlet)", fontsize=16)
plt.tight_layout()
plt.show()

**Come leggere il grafico.** Un pannello per periodo, con le 15 parole o bigrammi più distintivi del periodo rispetto al resto del corpus. La lunghezza della barra è lo z-score; la linea tratteggiata grigia è la soglia 1,96 (significatività al 5%): tutte le barre la superano ampiamente, quindi nessuna di queste differenze è casuale.

Da notare le barre del 2021-2024: più corte, perché il periodo ha solo 96 articoli e quindi meno evidenza statistica.

**Cosa emerge.**
- **2017-2020**: il lessico delle *famiglie* — *children, families, parents, separated, migrant children* — insieme a **undocumented immigrants** (z = 6,4), *sanctuary cities* e *wall*.
- **2021-2024**: il lessico del *confine* e dell'*asilo* — *asylum, southern border, guidelines, biden administration*.
- **2025-Presente**: il lessico dello *scontro sul territorio* — *crackdown, agents, judge, shooting, guard, protesters* e le città coinvolte (*minnesota, chicago*).

**Cosa cambia da un periodo al successivo.** Il confronto "periodo vs resto" dice cosa caratterizza un'epoca; il confronto tra periodi **consecutivi** isola invece le parole che entrano (z > 0) e quelle che escono (z < 0) nel passaggio da un'amministrazione all'altra.

In [ ]:
transizioni = list(zip(PERIODI[:-1], PERIODI[1:]))
righe = {}
for prima, dopo in transizioni:
    res = log_odds_dirichlet(counts_matrix, (df['periodo'] == dopo).values, (df['periodo'] == prima).values)
    res = res.sort_values("z")
    righe[f"{prima} → {dopo}: in crescita"] = [f"{r.term} ({r.z:.1f})" for r in res.tail(10).iloc[::-1].itertuples()]
    righe[f"{prima} → {dopo}: in calo"] = [f"{r.term} ({r.z:.1f})" for r in res.head(10).itertuples()]
pd.DataFrame(righe)

**Come vengono nominate le persone.** Il confronto per periodo fa emergere un cambio di etichetta: *illegal immigrants* è distintivo del 2003-2016, *undocumented immigrants* del 2017-2020. Tracciamo anno per anno la quota di articoli che usa ciascuna formula (conteggio per articolo, non per occorrenza, così un singolo articolo che ripete il termine non pesa di più).

In [ ]:
ETICHETTE_PERSONE = {
    "illegal immigrant(s)": r"\billegal immigrants?\b",
    "illegal alien(s)": r"\billegal aliens?\b",
    "undocumented immigrant(s)": r"\bundocumented immigrants?\b",
    "migrant(s)": r"\bmigrants?\b",
}
testo_lower = df['semantic_text'].str.lower()
etichette = pd.DataFrame({nome: testo_lower.str.contains(rx, regex=True) for nome, rx in ETICHETTE_PERSONE.items()})
etichette['anno'] = df['pub_date'].dt.year
quota_annua = etichette.groupby('anno').mean() * 100
n_annuo = etichette.groupby('anno').size()
quota_annua = quota_annua[n_annuo >= 10]  # anni con meno di 10 articoli: quote troppo instabili

fig, ax = plt.subplots(figsize=(15, 6))
for nome in ETICHETTE_PERSONE:
    ax.plot(quota_annua.index, quota_annua[nome], marker='o', linewidth=2, label=nome)
for anno, testo in [(2017, 'Trump I'), (2021, 'Biden'), (2025, 'Trump II')]:
    ax.axvline(x=anno, color='grey', linestyle=':', alpha=0.7)
    ax.text(anno + 0.1, ax.get_ylim()[1] * 0.95, testo, color='grey', fontsize=10)
ax.set_title("Come il NYT nomina le persone negli articoli ICE: quota di articoli per anno (anni con ≥10 articoli)", fontsize=14)
ax.set_ylabel("% articoli dell'anno")
ax.set_xlabel("Anno")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Articoli per anno inclusi:", n_annuo[n_annuo >= 10].to_dict())

- **Nel 2025-2026 tutte le formule crollano** (sotto il 12% e poi sotto il 4%): il soggetto degli articoli non sono più le persone migranti ma l'agenzia e le sue operazioni (*agents, crackdown, raids*), coerente con il grafico di distintività del 2025-Presente.

## 4. Metodi esplorati e scartati: t-SNE, Word2Vec, traiettorie del vocabolario

Nelle prime versioni del notebook c'erano tre analisi che sono state rimosse (2026-10-13) perché i loro risultati non sono affidabili su questo corpus. Le riportiamo qui per spiegare perché non compaiono nel report.

**t-SNE sulla matrice TF-IDF** (mappa 2D degli articoli, colorata per periodo e poi per topic).
- Era calcolato su TF-IDF ridotto a 50 dimensioni con SVD, che conservava solo il **17,6% della varianza**: la mappa rappresentava una piccola parte dell'informazione dei testi.
- t-SNE conserva solo le vicinanze **locali**: distanze tra gruppi, dimensioni dei gruppi e posizione sulla mappa non hanno significato. La mappa non permette quindi di dire che un periodo o un topic è "lontano" da un altro.
- La forma dipende dalla perplexity, scelta **a occhio** (50 perché "visivamente migliore"), e dal seed: è una visualizzazione, non una misura.
- Il raggruppamento per significato degli articoli è già fatto, e validato su più seed, da BERTopic (§5).

**Word2Vec** (parole più vicine a *ice*, sull'intero corpus e per periodo).
- Word2Vec impara il significato di una parola dai contesti in cui compare e ha bisogno di **milioni di parole**; il corpus ne ha **62.618** dopo la pulizia (2.093 testi brevi), e solo 2.030 parole superano la frequenza minima.
- **Modelli per periodo**: le similarità con *ice* sono tutte tra 0,98 e 1,00 e i vicini sono parole senza relazione (*promise, door, $, company* nel 2003-2016; *new, year, york* nel 2021-2024, che ha solo 96 articoli). Con così pochi dati tutti i vettori finiscono quasi nella stessa direzione: la similarità non distingue più niente.
- **Modello globale**: i vicini di *ice* (*clash, protest, newark, shooting, minneapolis*) sembrano sensati, ma sono un artefatto. Fino al 2016 il NYT scriveva quasi sempre il nome per esteso (il token *ice* compare nel 3% degli articoli 2003-2016) e dal 2025 usa l'acronimo (39%): *ice* compare quasi solo nei testi del 2025-2026, quindi i suoi vicini sono il lessico di quel periodo, non un significato della parola.

### 4.1 LDA a 4 topic

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd
import matplotlib.pyplot as plt

# 1. Definiamo la nostra blacklist di parole "standard" e vuote di significato specifico
parole_onnipresenti = [
    'ice', 'immigration', 'immigrant', 'immigrants', 'new', 'york', 
    'administration', 'say', 'said', 'trump', 'biden', 'obama', 
    'year', 'state', 'federal', 'official', 'government', 'president', 'agency',
    'states', 'city'
]

# 2. Configuriamo il vettorizzatore in modo molto più aggressivo
tf_vectorizer = CountVectorizer(
    max_df=0.50,               # NOVITÀ: ignora le parole presenti in più di metà del dataset
    min_df=5, 
    max_features=5000,
    stop_words=parole_onnipresenti  # NOVITÀ: applichiamo la blacklist
)

tf_matrix = tf_vectorizer.fit_transform(df['clean_text'])
tf_feature_names = tf_vectorizer.get_feature_names_out()

# 3. Addestriamo l'LDA (proviamo con 4 topic per vedere se si delineano meglio)
N_TOPICS = 4
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=15)
document_topics = lda.fit_transform(tf_matrix)

print("I 4 TOPIC LATENTI PURIFICATI DAL RUMORE DI FONDO:")
print("=" * 70)
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[:-11:-1]
    top_words = [tf_feature_names[i] for i in top_words_idx]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

## 5. Topic Modeling con BERTopic

Il log-odds (sezione 3) e la LDA (sezione 4.1) lavorano sulle singole parole. BERTopic invece raggruppa gli articoli per **significato della frase**, usando embedding di un modello transformer. Questo lo rende adatto a testi brevi come i nostri (headline + abstract + lead, ~50 parole per articolo), dove modelli classici come LDA faticano.

La pipeline di BERTopic:
1. **Embedding**: ogni articolo diventa un vettore da 768 dimensioni (modello `all-mpnet-base-v2`)
2. **UMAP**: riduzione a 5 dimensioni
3. **HDBSCAN**: clustering per densità; gli articoli che non stanno in nessun cluster finiscono nel topic `-1` (outlier)
4. **c-TF-IDF**: per ogni cluster estrae le parole più rappresentative, che diventano l'etichetta del topic

Usiamo `semantic_text` (non lemmatizzato): gli embedding funzionano meglio sulle frasi naturali che sui lemmi.

**Limite metodologico**: l'analisi è su titolo, abstract e lead paragraph, non sul testo integrale degli articoli.

### 5.1 Embedding

Modello `all-mpnet-base-v2` (768 dimensioni, legge fino a 384 token). Fino al 2026-10-13 il notebook usava `all-MiniLM-L6-v2` (384 dimensioni, 256 token): più veloce ma meno accurato nei benchmark di sentence-transformers. La sostituzione è fatta insieme nei notebook 04, 07 e 09, per tenere confrontabili i topic. Il confronto tra i due modelli sugli stessi articoli è nella §5.10.

Su `semantic_text` (mediana 74 parole fino al 2024, 34 dal 2025) nessun articolo arriva al limite di token, quindi il cambio non riguarda il troncamento.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-mpnet-base-v2"

docs = df['semantic_text'].tolist()

log.info(f"Calcolo embedding con {EMBEDDING_MODEL} su {len(docs)} articoli...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embedding_model.encode(docs, show_progress_bar=False)

print(f"Matrice embedding: {embeddings.shape[0]} articoli x {embeddings.shape[1]} dimensioni")

### 5.2 Addestramento

Parametri principali:
- `min_topic_size=15`: un topic deve contenere almeno 15 articoli
- `UMAP(random_state=42)`: senza seed fisso UMAP dà risultati diversi a ogni esecuzione
- `CountVectorizer(ngram_range=(1, 2))`: le etichette possono contenere bigrammi (es. "national guard", "sanctuary cities")

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP

# "ice" compare in quasi tutti gli articoli (come nel topic network, dove il nodo ICE è escluso):
# lasciarlo nelle etichette le renderebbe tutte uguali. Le altre sono parole di servizio del NYT
# ("old" viene dalle età, es. "34-year-old").
custom_stopwords = list(ENGLISH_STOP_WORDS | {"ice", "new", "said", "says", "mr", "ms", "year", "years", "old"})

vectorizer_model = CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 2), min_df=3)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=15,
)

log.info("Addestramento BERTopic...")
topics, _ = topic_model.fit_transform(docs, embeddings)

n_outliers = sum(t == -1 for t in topics)
print(f"Topic trovati: {len(set(topics)) - 1}")
print(f"Articoli outlier (topic -1): {n_outliers} ({n_outliers / len(docs):.1%})")

### 5.3 Riduzione degli outlier

HDBSCAN è prudente: su testi brevi lascia fuori molti articoli. Con `reduce_outliers` assegniamo ogni outlier al topic più simile, ma solo se la similarità coseno tra il suo embedding e quello del topic supera una soglia (0.4). Sotto soglia l'articolo resta `-1`: forzare anche quelli significherebbe assegnare articoli che non somigliano davvero a nessun topic.

In [ ]:
OUTLIER_THRESHOLD = 0.4

new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings, threshold=OUTLIER_THRESHOLD)
# Ricalcola le parole rappresentative sui topic aggiornati (altrimenti restano quelle pre-riassegnazione)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer_model)

df['topic'] = new_topics

n_outliers_after = sum(t == -1 for t in new_topics)
print(f"Outlier prima:  {n_outliers} ({n_outliers / len(docs):.1%})")
print(f"Outlier dopo:   {n_outliers_after} ({n_outliers_after / len(docs):.1%})")

### 5.4 I topic trovati

Per ogni topic: numero di articoli, parole più rappresentative e un **nome** breve, assegnato a mano leggendo le parole c-TF-IDF e un campione di titoli del topic. Il nome sostituisce le prime 3 parole nelle etichette dei grafici (`topic_labels`, es. `3: Detention centers`). I nomi sono in inglese, come le etichette dei grafici del notebook 10..

I numeri dei topic dipendono dall'addestramento: se cambiano dati, modello o parametri, lo stesso numero può indicare un tema diverso. Per questo ogni nome è legato anche alla **prima parola c-TF-IDF** attesa del topic: se non corrisponde, la cella si ferma con un errore invece di assegnare nomi sbagliati, e i nomi vanno rivisti.

In [ ]:
topic_info = topic_model.get_topic_info()

# numero del topic -> (prima parola c-TF-IDF attesa, nome). Nomi assegnati sul run mpnet del 2026-10-13
TOPIC_NAMES = {
    0: ("trump", "National politics: Trump and the parties"),
    1: ("local", "Local police and sanctuary cities"),
    2: ("shooting", "Shootings by agents"),
    3: ("detention", "Detention centers"),
    4: ("children", "Migrant children and families"),
    5: ("fear", "Raids and fear in communities"),
    6: ("security", "Homeland Security leadership"),
    7: ("migrants", "Deportations to third countries"),
    8: ("student", "Detained students and activists"),
    9: ("protests", "Protests in cities"),
    10: ("shutdown", "DHS funding and shutdown"),
    11: ("deported", "Individual deportation cases"),
    12: ("court", "Supreme Court and immigration policy"),
    13: ("protesters", "Protests at detention centers"),
    14: ("guard", "National Guard"),
    15: ("workers", "Workplace raids"),
    16: ("arrested", "Arrests and police operations"),
    17: ("judge", "Lawsuits"),
    18: ("guantánamo", "Guantánamo"),
    19: ("congress", "Congressional oversight of detention"),
    20: ("company", "Tech companies and ICE"),
    21: ("agents", "Agent masks and equipment"),
    22: ("chicago", "Crackdown in Chicago and other cities"),
    23: ("arrests", "ICE operations and arrests"),
    24: ("airports", "ICE at airports"),
}

found = sorted(set(topic_info['Topic']) - {-1})
mismatch = [t for t in found if t not in TOPIC_NAMES or topic_model.get_topic(t)[0][0] != TOPIC_NAMES[t][0]]
assert not mismatch and len(found) == len(TOPIC_NAMES), f"Topic cambiati rispetto ai nomi assegnati: {mismatch}. Rivedere TOPIC_NAMES."

topic_names = {t: name for t, (_, name) in TOPIC_NAMES.items()}
topic_names[-1] = "Outlier"
topic_labels = {t: f"{t}: {name}" for t, name in topic_names.items()}
df['topic_name'] = df['topic'].map(topic_names)
df['topic_label'] = df['topic'].map(topic_labels)

topic_table = topic_info[['Topic', 'Count']].copy()
topic_table['Nome'] = topic_table['Topic'].map(topic_names)
topic_table['Parole rappresentative'] = topic_table['Topic'].apply(
    lambda t: ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
)
pd.set_option('display.max_colwidth', 120)
display(topic_table)

Vediamo ora i titoli degli articoli più rappresentativi di ciascuno.

In [ ]:
for t in sorted(topic_labels):
    if t == -1:
        continue
    print(f"\n{topic_labels[t]}  ({(df['topic'] == t).sum()} articoli)")
    print("-" * 70)
    for doc in topic_model.get_representative_docs(t)[:3]:
        print(f"  • {doc[:130]}...")

### 5.5 Parole principali per topic

Tutti i topic, ordinati per numero di articoli **dopo** la riassegnazione degli outlier (§5.3). Per ogni topic le 8 parole con il punteggio c-TF-IDF più alto.

In [ ]:
# Ordine per dimensione finale: i numeri di BERTopic seguono la dimensione prima di reduce_outliers
topics_by_size = df.loc[df['topic'] != -1, 'topic'].value_counts().index.tolist()

N_COLS = 5
n_rows = -(-len(topics_by_size) // N_COLS)
fig, axes = plt.subplots(nrows=n_rows, ncols=N_COLS, figsize=(4.4 * N_COLS, 2.9 * n_rows))
colors = sns.color_palette("husl", len(topics_by_size))

for ax, t, color in zip(axes.reshape(-1), topics_by_size, colors):
    words, scores = zip(*topic_model.get_topic(t)[:8])
    ax.barh(words[::-1], scores[::-1], color=color)
    ax.set_title(f"Topic {t} ({(df['topic'] == t).sum()} art.)", fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=9)
    ax.set_xlabel("c-TF-IDF", fontsize=8)
for ax in axes.reshape(-1)[len(topics_by_size):]:
    ax.axis("off")

plt.suptitle("Parole più rappresentative per topic (c-TF-IDF), topic ordinati per numero di articoli", fontsize=16)
plt.tight_layout()
plt.show()

**Come leggere il grafico.** Un pannello per topic, dal più grande al più piccolo; nel titolo il numero del topic e i suoi articoli. Le barre sono le 8 parole (o bigrammi) che meglio caratterizzano il topic.

**Cosa indica l'asse orizzontale.** È il punteggio **c-TF-IDF** (class-based TF-IDF), con cui BERTopic sceglie le parole di un topic. Tutti gli articoli di un topic vengono trattati come un unico documento e per ogni parola si moltiplicano due fattori:
- **frequenza nel topic**: quante volte la parola compare negli articoli del topic, divisa per il numero totale di parole del topic (quindi una quota, per questo i valori sono piccoli, 0,01-0,1);
- **rarità negli altri topic**: `log(1 + A / f)`, dove `f` è quante volte la parola compare in tutto il corpus e `A` il numero medio di parole per topic. Una parola presente ovunque (es. *immigration*) ha un peso basso anche se è frequente.

Un punteggio alto vuol dire una parola frequente nel topic e relativamente rara negli altri. Il valore va letto **dentro** un pannello, per ordinare le parole del topic: tra topic diversi non è confrontabile, perché dipende dalla lunghezza totale del topic e dal suo vocabolario.

### 5.6 Composizione dei topic per periodo

I periodi hanno dimensioni molto diverse (il 2025-Presente da solo ha più articoli di tutti gli altri insieme), quindi i conteggi assoluti sarebbero dominati dall'ultimo periodo. Normalizziamo per periodo: ogni colonna mostra **che quota degli articoli di quel periodo** appartiene a ciascun topic.

In [ ]:
topic_period = pd.crosstab(df['topic_label'], df['periodo'], normalize='columns') * 100
topic_period = topic_period.drop(index=topic_labels[-1], errors='ignore')
topic_period = topic_period.loc[topic_period.max(axis=1).sort_values(ascending=False).index]

print("Articoli per periodo:")
print(df['periodo'].value_counts().sort_index().to_string())

plt.figure(figsize=(12, 14))
sns.heatmap(topic_period, annot=True, fmt=".1f", cmap="Blues", cbar_kws={'label': '% articoli del periodo'})
plt.title("Peso di ciascun topic per periodo (%)", fontsize=16)
plt.xlabel("Periodo")
plt.ylabel("Topic")
plt.tight_layout()
plt.show()

### 5.7 Evoluzione dei topic nel tempo

Conteggio annuale degli articoli per i topic più grandi, con `topics_over_time` di BERTopic.

In [ ]:
df['anno'] = df['pub_date'].dt.year

topics_over_time = topic_model.topics_over_time(docs, df['anno'].tolist(), nr_bins=None)

# topics_over_time omette gli anni con 0 articoli: senza reindex la linea unirebbe
# direttamente gli anni non vuoti, facendo sembrare graduale un salto
anni_completi = range(df['anno'].min(), df['anno'].max() + 1)
trend = (
    topics_over_time[topics_over_time['Topic'].isin(topics_by_size[:8])]
    .pivot(index='Timestamp', columns='Topic', values='Frequency')
    .reindex(anni_completi, fill_value=0)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(16, 8))
colors = sns.color_palette('tab10', 8)
for t, color in zip(topics_by_size[:8], colors):
    ax.plot(trend.index, trend[t], marker='o', linewidth=2, color=color, label=topic_labels[t])

for anno, testo in [(2017, 'Inizio Era Trump I'), (2021, 'Inizio Era Biden'), (2025, 'Inizio Era Trump II')]:
    ax.axvline(x=anno, color='grey', linestyle=':', alpha=0.7)
    ax.text(anno + 0.1, ax.get_ylim()[1] * 0.95, testo, color='grey', fontsize=10)

ultimo_mese = df['pub_date'].max().strftime('%Y-%m')
ax.set_xticks(list(anni_completi))
ax.set_xticklabels(anni_completi, rotation=45)
ax.set_title(f"Evoluzione annuale dei principali topic ICE sul NYT (ultimo anno parziale, dati fino a {ultimo_mese})", fontsize=16)
ax.set_xlabel("Anno")
ax.set_ylabel("Numero di articoli")
ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 5.9 Stabilità su più seed

UMAP è stocastico: con un seed diverso la mappa a 5 dimensioni cambia e HDBSCAN può trovare gruppi diversi. Sul Guardian (`07_guardianTopicModeling.ipynb`) il testo breve è risultato instabile: da 3 a 22 topic a seconda del seed, con 2 collassi su 5 in un unico topic gigante. Il modello qui sopra usa testo breve e un solo seed (42), quindi va verificato che i suoi topic non dipendano dal caso.

Riaddestriamo lo stesso modello, con gli stessi parametri, su `STABILITY_SEEDS` (i 5 del Guardian più altri 5) e misuriamo:

- **numero di topic** e **topic più grande** per seed (un topic oltre il 50% del corpus è un collasso);
- **AMI** (Adjusted Mutual Information) tra ogni coppia di seed: 1 = stessi gruppi, 0 = gruppi indipendenti. Calcolata due volte: su tutti gli articoli dopo la riassegnazione degli outlier, e solo sugli articoli che HDBSCAN ha messo in un cluster in entrambi i seed (il nucleo dei topic, senza gli articoli riassegnati).

Il modello di riferimento resta quello con seed 42 addestrato sopra: non viene riaddestrato.

In [ ]:
from itertools import combinations
from sklearn.metrics import adjusted_mutual_info_score

REFERENCE_SEED = 42
STABILITY_SEEDS = [0, 1, 7, 42, 123, 2, 3, 5, 11, 99]
OUT_TOPIC_STABILITY = PROCESSED_DIR / "ice_topic_stability.csv"

logging.getLogger("BERTopic").setLevel(logging.WARNING)


def fit_topics_seed(seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Stessa pipeline di 5.2-5.3 con un altro seed UMAP. Ritorna (topic HDBSCAN, topic dopo reduce_outliers)."""
    model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        vectorizer_model=CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 2), min_df=3),
        min_topic_size=15,
    )
    raw, _ = model.fit_transform(docs, embeddings)
    raw = np.array(raw)
    final = raw
    if (raw == -1).any():
        final = np.array(model.reduce_outliers(docs, raw, strategy="embeddings", embeddings=embeddings, threshold=OUTLIER_THRESHOLD))
    return raw, final


seed_runs = {REFERENCE_SEED: (np.array(topics), np.array(new_topics))}
for seed in STABILITY_SEEDS:
    if seed not in seed_runs:
        seed_runs[seed] = fit_topics_seed(seed)

seed_table = pd.DataFrame([
    {
        "seed": seed,
        "topic": len(set(final) - {-1}),
        "outlier prima %": 100 * (raw == -1).mean(),
        "outlier dopo %": 100 * (final == -1).mean(),
        "topic più grande %": 100 * pd.Series(final[final != -1]).value_counts().max() / len(final),
    }
    for seed, (raw, final) in ((s, seed_runs[s]) for s in STABILITY_SEEDS)
]).set_index("seed")
display(seed_table.round(1))

ami_all, ami_core = [], []
for a, b in combinations(STABILITY_SEEDS, 2):
    ami_all.append(adjusted_mutual_info_score(seed_runs[a][1], seed_runs[b][1]))
    core = (seed_runs[a][0] != -1) & (seed_runs[b][0] != -1)
    ami_core.append(adjusted_mutual_info_score(seed_runs[a][0][core], seed_runs[b][0][core]))

print(f"Topic per seed: {seed_table['topic'].min()}-{seed_table['topic'].max()} (riferimento, seed {REFERENCE_SEED}: {seed_table.loc[REFERENCE_SEED, 'topic']})")
print(f"Collassi (un topic > 50% del corpus): {(seed_table['topic più grande %'] > 50).sum()}/{len(STABILITY_SEEDS)}")
print(f"AMI tra seed, tutti gli articoli:     media {np.mean(ami_all):.3f} (min {min(ami_all):.3f}, max {max(ami_all):.3f})")
print(f"AMI tra seed, solo nucleo dei topic:  media {np.mean(ami_core):.3f} (min {min(ami_core):.3f}, max {max(ami_core):.3f})")

La tabella mostra i risultati delle esecuzioni sui 10 diversi seed. Per ogni seed mostra il numero di topic trovati, la percentuale di outlier prima e dopo la riassegnazione, e il peso del topic più grande ( > 50% segnala un collasso).

Viene poi calcolato l'AMI fra coppie di seed (quindi modi diversi di dividere i topic): 1 indica stessi gruppi (previsione perfetta), 0 gruppi indipendenti. In particolare l'AMI è calcolata due volte: prima su tutti gli articoli dopo l'assegnazione degli outlier e poi sul nucleo (cioè sugli articoli che HDBSCAN ha messo in un cluster in entrambi i seed, esclusi i riassegnati).

Un valore di 0.852 (su tutti gli articoli) indica che i topic coincidono in larga parte da un seed all'altro. 0.982 sul nucleo indica due previsioni quasi identiche sul cluster maggiore.

**Stabilità dei singoli topic del modello di riferimento.** L'AMI è una media su tutto il modello; qui guardiamo topic per topic. Per ogni topic del seed 42 cerchiamo, in ciascun altro seed, il topic che gli somiglia di più e misuriamo la sovrapposizione con l'indice di Jaccard (articoli in comune / articoli nell'uno o nell'altro: 1 = stesso insieme di articoli). Un topic con Jaccard medio alto ricompare quasi identico a ogni seed; uno basso viene spezzato o fuso in modo diverso ogni volta e va interpretato con cautela.

`riassegnati %` è la quota di articoli del topic che HDBSCAN aveva lasciato fuori e che `reduce_outliers` ha poi assegnato.

In [ ]:
STABLE_JACCARD = 0.7

ref_raw, ref_final = seed_runs[REFERENCE_SEED]
rows = []
for t in sorted(set(ref_final) - {-1}):
    members = np.where(ref_final == t)[0]
    member_set = set(members)
    scores = []
    for seed in STABILITY_SEEDS:
        if seed == REFERENCE_SEED:
            continue
        other = seed_runs[seed][1]
        best = 0.0
        for u in set(other[members]) - {-1}:
            other_set = set(np.where(other == u)[0])
            best = max(best, len(member_set & other_set) / len(member_set | other_set))
        scores.append(best)
    rows.append({
        "topic": topic_labels[t],
        "articoli": len(members),
        "riassegnati %": round(100 * (ref_raw[members] == -1).mean()),
        "jaccard medio": round(float(np.mean(scores)), 3),
        "jaccard minimo": round(float(min(scores)), 2),
        "stabile": np.mean(scores) >= STABLE_JACCARD,
    })

topic_stability = pd.DataFrame(rows).sort_values("jaccard medio", ascending=False).reset_index(drop=True)
topic_stability.to_csv(OUT_TOPIC_STABILITY, index=False)
log.info("Scritto %s", OUT_TOPIC_STABILITY.relative_to(BASE_DIR))

n_stable = int(topic_stability["stabile"].sum())
print(f"Topic stabili (Jaccard medio >= {STABLE_JACCARD}): {n_stable}/{len(topic_stability)}, "
      f"{topic_stability.loc[topic_stability['stabile'], 'articoli'].sum() / len(df):.0%} degli articoli")
display(topic_stability)

- **Il modello NYT è stabile.** Su 10 seed: 23-28 topic, nessun collasso, AMI medio 0,80. Sul nucleo dei topic (articoli che HDBSCAN mette in un cluster) l'AMI sale a 0,96: i gruppi di base sono quasi identici a ogni seed. L'instabilità trovata sul testo breve del Guardian (AMI 0,39, 2 collassi su 5) qui non si ripresenta.
- **La variabilità viene dagli articoli riassegnati.** Il 36-45% degli articoli è outlier prima di `reduce_outliers`; sono loro a cambiare topic tra un seed e l'altro, e a far scendere l'AMI da 0,96 a 0,80.
- **Il seed 42 è il più prudente**: 23 topic, il minimo tra i 10 seed. Gli altri seed spezzano in più parti alcuni topic grandi.
- **18 topic su 23 sono stabili** (Jaccard medio ≥ 0,7) e coprono il 69% degli articoli. Da interpretare con cautela: **5** (giudici e tribunali, 0,40: il meno affidabile), **1** (Trump e deportazioni, 0,63), **15** (bambini e famiglie migranti, 0,66, con il 65% di articoli riassegnati), **18** e **19** (appena sotto soglia). I topic 1 e 5 sono grandi e generici: a seconda del seed vengono divisi o fusi con i vicini.
- I topic legati a un evento o a un luogo preciso (Guardia Nazionale, Guantánamo, proteste a Los Angeles, città santuario) sono i più solidi.

Limiti: la soglia 0,7 è una convenzione; il Jaccard minimo mostra che anche un topic stabile in media può sparire in un singolo seed (topic 20, studenti: 0,06 in un seed su 9).

### 5.10 Dataset degli articoli con il topic

Salviamo `articles_ice_topics.parquet`: tutte le colonne di `articles_ice.parquet` più il topic di ogni articolo, da usare nelle analisi successive (sentiment per topic, confronto con i commenti) senza riaddestrare BERTopic.

| colonna | contenuto |
|---|---|
| `topic` | numero del topic dopo la riassegnazione degli outlier (§5.3); -1 = outlier rimasto |
| `topic_name` | nome del topic (§5.4) |
| `topic_words` | prime 5 parole c-TF-IDF |
| `topic_was_outlier` | l'articolo era outlier per HDBSCAN ed è stato assegnato da `reduce_outliers`: assegnazione più debole, è la parte che cambia tra seed (§5.9) |
| `topic_stable` | il topic ha Jaccard medio ≥ 0,7 tra seed (§5.9) |

Il file va rigenerato insieme al notebook: i numeri e i nomi valgono solo per questo addestramento.

In [ ]:
OUT_ARTICLES_TOPICS = PROCESSED_DIR / "articles_ice_topics.parquet"

topic_words = {t: ", ".join(w for w, _ in topic_model.get_topic(t)[:5]) for t in topic_names if t != -1}
stable_by_label = dict(zip(topic_stability['topic'], topic_stability['stabile']))

articles_topics = pd.read_parquet(IN_ICE).merge(
    pd.DataFrame({
        'id': df['id'],
        'topic': df['topic'],
        'topic_name': df['topic_name'],
        'topic_words': df['topic'].map(topic_words),
        'topic_was_outlier': np.array(topics) == -1,
        'topic_stable': df['topic_label'].map(stable_by_label).fillna(False).astype(bool),
    }),
    on='id', how='left', validate='one_to_one',
)
assert articles_topics['topic'].notna().all()
articles_topics.to_parquet(OUT_ARTICLES_TOPICS, index=False)
log.info("Scritto %s (%d articoli)", OUT_ARTICLES_TOPICS.relative_to(BASE_DIR), len(articles_topics))

display(articles_topics.groupby(['topic', 'topic_name'])
        .agg(articoli=('id', 'size'), riassegnati=('topic_was_outlier', 'mean'), stabile=('topic_stable', 'first'))
        .assign(riassegnati=lambda x: (x['riassegnati'] * 100).round(0).astype(int).astype(str) + '%')
        .sort_values('articoli', ascending=False))

**Come leggere la tabella.** Una riga per topic, dal più grande: articoli, quota di articoli riassegnati da `reduce_outliers` e stabilità tra seed. I topic con molti riassegnati o non stabili (es. 16 e 23, arresti generici) vanno usati con cautela nelle analisi per topic.